In [4]:
# Notebook: Q1_NAWG_Validation_v01.ipynb | Cell 1
# Load model outputs + NAWG severity table, match upazila names, build upazila table.
from pathlib import Path
import re
import numpy as np
import pandas as pd

# ---- Settings ----
ROOT = Path(r"D:\Siam\Thesis_GeoAI")
MASTER = ROOT / "04_master_table" / "master_v06_20261020.csv"
HAZ_V03 = ROOT / "05_models" / "final_v03" / "hazard_predictions_v03.csv"
HAZ_V02 = ROOT / "05_models" / "final_v02" / "hazard_predictions_v02.csv"
LABELS = ROOT / "03_features" / "Label_Flooded_Event_v02.csv"
NAWG_XLSX = ROOT / "01_raw" / "nawg_remal" / "Severity_Analysis_and_Ranking_of_Affected_Upazila_Cyclone_REMAL_2024.xlsx"
OUT_DIR = ROOT / "05_models" / "q1_nawg_v01"
UP_COL, DI_COL = "Upazila", "District"          # fixed after inspecting master columns
OUR_DISTRICTS = ["barguna", "patuakhali", "bhola", "pirojpur", "bagerhat"]
EXPECTED_NAWG_ROWS = 23
N_MODEL_CELLS = 160874
# Spelling variants -> one canonical key (lowercase, letters only, applied to both sides)
CANON = {"sharankhola": "sarankhola", "morelganj": "morrelganj", "monpura": "manpura",
         "indurkani": "zianagar", "pathorghata": "patharghata", "borhanuddin": "burhanuddin",
         "borhanuddinupazila": "burhanuddin"}

OUT_DIR.mkdir(parents=True, exist_ok=True)
for p in [MASTER, HAZ_V03, HAZ_V02, LABELS, NAWG_XLSX]:
    assert p.exists(), f"Missing file: {p}"

def key(s):
    k = re.sub(r"[^a-z]", "", str(s).lower())
    return CANON.get(k, k)

def pick_prob(df, tag):
    num = [c for c in df.columns if c != "Grid_ID" and pd.api.types.is_numeric_dtype(df[c])]
    prob = [c for c in num if df[c].dropna().between(0, 1).all()
            and not re.search(r"rank|pct|sd|(^|_)fr(_|$)|freq|aoa|flag|(^|_)di($|_)", c.lower())]
    lg = [c for c in prob if "lgbm" in c.lower()]
    cand = lg if lg else prob
    assert len(cand) == 1, f"[{tag}] cannot pick hazard prob column; candidates {cand}. Columns: {list(df.columns)}"
    return cand[0]

# ---- Master (model cells, population, upazila, district) ----
m = pd.read_csv(MASTER, usecols=["Grid_ID", "Is_Water", "Population_Count", UP_COL, DI_COL, "Upazila_Name"])
assert m["Grid_ID"].is_unique, "Duplicate Grid_ID in master"
m = m[m["Is_Water"] == 0].copy()
both = m[[UP_COL, "Upazila_Name"]].dropna()
agree = (both[UP_COL].map(key) == both["Upazila_Name"].map(key)).mean()
print(f"Unique: {UP_COL} {m[UP_COL].nunique()} | Upazila_Name {m['Upazila_Name'].nunique()} | name agreement {agree:.1%}")
print(f"NaN {UP_COL}: {int(m[UP_COL].isna().sum())} cells | NaN {DI_COL}: {int(m[DI_COL].isna().sum())} cells")
assert m[UP_COL].notna().all() and m[DI_COL].notna().all(), "Cells without upazila/district: paste this output"
print(f"Model cells: {len(m):,}")
assert len(m) == N_MODEL_CELLS
print("Population NaN cells:", int(m["Population_Count"].isna().sum()))
m["Population_Count"] = m["Population_Count"].fillna(0)

# ---- Hazard predictions ----
h3, h2 = pd.read_csv(HAZ_V03), pd.read_csv(HAZ_V02)
c3, c2 = pick_prob(h3, "v03"), pick_prob(h2, "v02")
fr = [c for c in h3.columns if pd.api.types.is_numeric_dtype(h3[c])
      and re.search(r"(^|_)fr(_|$)|freq", c.lower())]
c_fr = fr[0] if len(fr) == 1 else None
print(f"v03 column: {c3} | v02 column: {c2} | FR column: {c_fr}")
cells = m.merge(h3[["Grid_ID", c3]].rename(columns={c3: "P_v03"}), on="Grid_ID", how="left")
cells = cells.merge(h2[["Grid_ID", c2]].rename(columns={c2: "P_v02"}), on="Grid_ID", how="left")
if c_fr:
    cells = cells.merge(h3[["Grid_ID", c_fr]].rename(columns={c_fr: "P_FR"}), on="Grid_ID", how="left")
for c in [x for x in ["P_v03", "P_v02", "P_FR"] if x in cells]:
    assert cells[c].notna().all(), f"NaN in {c}"
    cells[c.replace("P_", "Pct_")] = (cells[c].rank(method="average") - 0.5) / len(cells)  # mid-rank, as Phase 5

# ---- SAR Remal labels (NaN = unobservable) ----
lab = pd.read_csv(LABELS, usecols=["Grid_ID", "Event_ID", "Flooded_Event"])
print("Event IDs in label file:", sorted(lab["Event_ID"].astype(str).unique()))
rem = lab[lab["Event_ID"].astype(str).str.contains("remal", case=False)]
assert len(rem) > 0 and rem["Grid_ID"].is_unique
cells = cells.merge(rem[["Grid_ID", "Flooded_Event"]].rename(columns={"Flooded_Event": "SAR_Remal"}),
                    on="Grid_ID", how="left")

# ---- Upazila aggregation (population-weighted) ----
def wmean(g, c):
    w = g["Population_Count"]
    return np.average(g[c], weights=w) if w.sum() > 0 else np.nan

rows = []
for (d, u), g in cells.groupby([DI_COL, UP_COL]):
    r = {"District": d, "Upazila": u, "Key": key(u), "Population_Total": g["Population_Count"].sum(),
         "N_Cells": len(g), "Hazard_Pct_PopW_v03": wmean(g, "Pct_v03"), "Hazard_Pct_PopW_v02": wmean(g, "Pct_v02"),
         "SAR_Remal_Flooded_Frac": g["SAR_Remal"].mean(), "SAR_Remal_N_Observable": int(g["SAR_Remal"].notna().sum())}
    if "Pct_FR" in cells:
        r["Hazard_Pct_PopW_FR"] = wmean(g, "Pct_FR")
    rows.append(r)
up = pd.DataFrame(rows)
print("Our upazilas:", len(up))
assert len(up) == 37 and up["Key"].is_unique

# ---- NAWG table + name matching ----
nw = pd.read_excel(NAWG_XLSX)
nw.columns = [str(c).strip() for c in nw.columns]
need = ["DISTRICT", "UPAZILA", "Impact of Crisis", "Condition of People Affected",
        "Complexity of Crisis", "Severity Option", "Severity Category"]
assert all(c in nw.columns for c in need), f"NAWG columns: {list(nw.columns)}"
nw = nw[nw["DISTRICT"].astype(str).str.strip().str.lower().isin(OUR_DISTRICTS)].copy()
print("NAWG rows in our 5 districts:", len(nw))
assert len(nw) == EXPECTED_NAWG_ROWS
nw["Key"] = nw["UPAZILA"].map(key)
nw = nw.rename(columns={"Impact of Crisis": "NAWG_Impact_Score", "Condition of People Affected": "NAWG_Condition_Score",
                        "Complexity of Crisis": "NAWG_Complexity_Score", "Severity Option": "NAWG_Severity_Score",
                        "Severity Category": "NAWG_Severity_Category", "UPAZILA": "NAWG_Upazila_Name",
                        "DISTRICT": "NAWG_District"})
keep = ["Key", "NAWG_District", "NAWG_Upazila_Name", "NAWG_Impact_Score", "NAWG_Condition_Score",
        "NAWG_Complexity_Score", "NAWG_Severity_Score", "NAWG_Severity_Category"]
t = up.merge(nw[keep], on="Key", how="left")
unmatched = set(nw["Key"]) - set(up["Key"])
print("Unmatched NAWG names:", unmatched if unmatched else "none")
if unmatched:
    print("Our Bhola upazilas:", sorted(up.loc[up["District"].str.lower() == "bhola", "Upazila"]))
assert not unmatched, "Name mismatch: paste this output"
mt = t.dropna(subset=["NAWG_Impact_Score"])
assert (mt["NAWG_District"].str.lower().str.strip() == mt["District"].astype(str).str.lower().str.strip()).all(), \
    "District mismatch in matched names"
print(f"\nMatched upazilas: {len(mt)} of 37")
print(mt[["District", "Upazila", "NAWG_Upazila_Name", "NAWG_Impact_Score", "Population_Total",
          "Hazard_Pct_PopW_v03", "Hazard_Pct_PopW_v02", "SAR_Remal_Flooded_Frac"]].round(3).to_string(index=False))
t.to_csv(OUT_DIR / "nawg_upazila_table_v01.csv", index=False)
print("\nSaved:", OUT_DIR / "nawg_upazila_table_v01.csv")

Unique: Upazila 37 | Upazila_Name 37 | name agreement 100.0%
NaN Upazila: 0 cells | NaN District: 0 cells
Model cells: 160,874
Population NaN cells: 0
v03 column: Hazard_Prob_LGBM | v02 column: Hazard_Prob_LGBM | FR column: Hazard_Score_FR
Event IDs in label file: ['Amphan', 'Remal', 'Yaas']
Our upazilas: 37
NAWG rows in our 5 districts: 23
Unmatched NAWG names: none

Matched upazilas: 23 of 37
  District        Upazila NAWG_Upazila_Name  NAWG_Impact_Score  Population_Total  Hazard_Pct_PopW_v03  Hazard_Pct_PopW_v02  SAR_Remal_Flooded_Frac
  Bagerhat         Mongla            Mongla                2.6        180054.295                0.265                0.461                   0.014
  Bagerhat      Morelganj        Morrelganj                2.9        330472.478                0.562                0.433                   0.486
  Bagerhat         Rampal            Rampal                2.4        204253.139                0.309                0.424                   0.164
  Bagerhat    

In [5]:
# Notebook: Q1_NAWG_Validation_v01.ipynb | Cell 2
# Frozen rules: Spearman(NAWG Impact, pop-weighted hazard pct), bootstrap over upazilas.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr

# ---- Settings ----
OUT_DIR = Path(r"D:\Siam\Thesis_GeoAI\05_models\q1_nawg_v01")
SEED, N_BOOT, N_EXPECTED = 42, 5000, 23

t = pd.read_csv(OUT_DIR / "nawg_upazila_table_v01.csv")
v = t.dropna(subset=["NAWG_Impact_Score"]).reset_index(drop=True)
assert len(v) == N_EXPECTED
y = v["NAWG_Impact_Score"].to_numpy()
models = {"v03": "Hazard_Pct_PopW_v03", "v02": "Hazard_Pct_PopW_v02"}
if "Hazard_Pct_PopW_FR" in v:
    models["FR"] = "Hazard_Pct_PopW_FR"
models["SAR_Remal_obs"] = "SAR_Remal_Flooded_Frac"   # secondary: label credibility

def rho(a, b):
    return spearmanr(a, b).correlation

rng = np.random.default_rng(SEED)
idx = rng.integers(0, len(v), size=(N_BOOT, len(v)))   # paired resamples for all models
boot = {k: np.array([rho(y[i], v[c].to_numpy()[i]) for i in idx]) for k, c in models.items()}

res = []
for k, c in models.items():
    b = boot[k]
    res.append({"Model": k, "Spearman_Rho": rho(y, v[c]), "CI95_Low": np.nanpercentile(b, 2.5),
                "CI95_High": np.nanpercentile(b, 97.5), "Boot_NaN": int(np.isnan(b).sum())})
res = pd.DataFrame(res)
d = boot["v03"] - boot["v02"]
diff = {"Diff_v03_minus_v02": rho(y, v[models["v03"]]) - rho(y, v[models["v02"]]),
        "CI95_Low": np.nanpercentile(d, 2.5), "CI95_High": np.nanpercentile(d, 97.5)}

def partial_spearman(x, yy, z):
    rx, ry, rz = (pd.Series(a).rank().to_numpy() for a in (x, yy, z))
    Z = np.column_stack([np.ones_like(rz), rz])
    ex = rx - Z @ np.linalg.lstsq(Z, rx, rcond=None)[0]
    ey = ry - Z @ np.linalg.lstsq(Z, ry, rcond=None)[0]
    return pearsonr(ex, ey)[0]

pop = v["Population_Total"].to_numpy()
partial = {k: partial_spearman(v[models[k]], y, pop) for k in ["v03", "v02"]}
sev_desc = rho(v["NAWG_Severity_Score"], v[models["v03"]])

print("PRIMARY: Spearman(NAWG Impact of Crisis, pop-weighted hazard pct), n =", len(v))
print(res.round(3).to_string(index=False))
print("\nDifference v03 - v02:", {k: round(x, 3) for k, x in diff.items()})
print("Partial Spearman controlling population (secondary):", {k: round(x, 3) for k, x in partial.items()})
print("Descriptive only: Spearman(NAWG overall severity, v03) =", round(sev_desc, 3))

r3 = res.set_index("Model").loc["v03"]
supportive = (r3["Spearman_Rho"] > 0) and (r3["CI95_Low"] > 0) and (diff["Diff_v03_minus_v02"] >= 0)
clearly_better = supportive and (diff["CI95_Low"] > 0)
print("\nVERDICT (frozen rules):")
print("  Supportive:", supportive, "| Clearly better than v02:", clearly_better)
if not supportive:
    print("  -> Report honestly: correction not confirmed by independent survey-based evidence.")

res.to_csv(OUT_DIR / "nawg_spearman_v01.csv", index=False)
with open(OUT_DIR / "nawg_config_v01.json", "w") as f:
    json.dump({"seed": SEED, "n_boot": N_BOOT, "n_upazila": len(v), "nawg_variable": "Impact of Crisis",
               "model_variable": "population-weighted mid-rank hazard percentile", "diff": diff,
               "partial_spearman_pop": partial, "supportive": bool(supportive),
               "clearly_better": bool(clearly_better)}, f, indent=2, default=float)
print("Saved results + config in", OUT_DIR)

PRIMARY: Spearman(NAWG Impact of Crisis, pop-weighted hazard pct), n = 23
        Model  Spearman_Rho  CI95_Low  CI95_High  Boot_NaN
          v03        -0.363    -0.694      0.053         0
          v02        -0.632    -0.841     -0.237         0
           FR        -0.183    -0.544      0.230         0
SAR_Remal_obs         0.383    -0.055      0.731         0

Difference v03 - v02: {'Diff_v03_minus_v02': np.float64(0.269), 'CI95_Low': np.float64(-0.045), 'CI95_High': np.float64(0.635)}
Partial Spearman controlling population (secondary): {'v03': np.float64(-0.365), 'v02': np.float64(-0.629)}
Descriptive only: Spearman(NAWG overall severity, v03) = -0.038

VERDICT (frozen rules):
  Supportive: False | Clearly better than v02: False
  -> Report honestly: correction not confirmed by independent survey-based evidence.
Saved results + config in D:\Siam\Thesis_GeoAI\05_models\q1_nawg_v01
